In [ ]:
import os
import re
import time
import argparse
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


In [ ]:
load_dotenv()

BASE_DIR   = Path.cwd().parent
CHROMA_DIR = BASE_DIR / "data" / "chroma_db"
COLLECTION = "babyos_youtube"


In [ ]:
CURATION_RULES = {
    "nhs": {
        "channel_keywords": ["NHS", "UK NHS", "National Health Service"],
        "topics": {
            "labour": [
                "signs of labour",
                "what to expect labour",
                "early labour symptoms"
            ],
            "pregnancy_overview": [
                "pregnancy week by week",
                "antenatal care explained"
            ],
            "postpartum": [
                "postnatal depression",
                "postnatal recovery"
            ]
        }
    }
}

In [ ]:
def build_queries(source, topic):
    rules = CURATION_RULES[source]
    return [
        f"{q} {source}" for q in rules["topics"][topic]
    ]

In [ ]:
def youtube_search(query):
    return api.search(
        q=query,
        type="video",
        order="relevance",
        videoDuration="medium",
        maxResults=5
    )

In [ ]:
def score(video, source_keywords):
    score = 0

    # channel authority boost
    if any(k.lower() in video.channelTitle.lower()
           for k in source_keywords):
        score += 3

    # title relevance
    score += similarity(video.title, query)

    # recency boost (soft)
    score += recency_decay(video.publishedAt)

    # engagement proxy (optional)
    score += log(video.viewCount + 1)

    return score

In [ ]:
def is_stable(video):
    return (
        video.duration_seconds > 120 and
        video.duration_seconds < 1800 and
        video.liveBroadcastContent == "none"
    )

In [ ]:
def dedupe(videos):
    seen = set()
    out = []

    for v in videos:
        key = (v.channelId, normalize(v.title))
        if key not in seen:
            seen.add(key)
            out.append(v)

    return out

In [ ]:
def build_registry(source, topic):
    queries = build_queries(source, topic)

    results = []
    for q in queries:
        vids = youtube_search(q)

        for v in vids:
            if is_stable(v):
                results.append(v)

    ranked = sorted(
        results,
        key=lambda v: score(v, CURATION_RULES[source]["channel_keywords"]),
        reverse=True
    )

    return ranked[:5]